In [12]:
from collections import Counter, defaultdict
from matplotlib.lines import Line2D
from scipy.spatial import distance
import matplotlib.pyplot as plt
from rdkit import Chem
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
import os
import tarfile
import pymol
from pymol import cmd
import matplotlib.image as mpimg
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen
from rdkit.Chem.QED import qed
import matplotlib.patches as patches
from pycirclize import Circos
from itertools import combinations

In [2]:
DOCKING_RESULTS_ORIGINAL = {}

# Define some paths
root = '../../../Documents_GPU/mtb-targeted-protein-degradation/notebooks'
PATH_TO_DOCKING_RESULTS_ORIGINAL = os.path.join(root, "..", "processed", "unidock_docking", 'docking_results')
PATH_TO_DOCKING_RESULTS_REAL_2 = os.path.join(root, "..", "processed", "unidock_REAL_docking_2", 'docking_results')

# Mapping IDs to SMILES
ID_TO_SMILES = pickle.load(open(os.path.join(root, "..", "processed", "enamine_characterization", "ID_TO_SMI.pkl"), "rb"))

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))
pocket_detection_data_interpro = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data_interpro.tsv"), sep='\t')

# Uniprot to gene name
uniprot_to_gene = pd.read_csv(os.path.join(root, "..", "data", "mtb_trna_synthetases_bosch_2021_fig5.csv"))
uniprot_to_gene = {i: j for i,j in zip(uniprot_to_gene['uniprot_ac'], uniprot_to_gene['gene_name_in_bosch_2021'])}

# For each pocket - ORIGINAL
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_ORIGINAL))):
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_ORIGINAL, pocket, 'report.csv'))
    DOCKING_RESULTS_ORIGINAL[pocket] = {i: j for i, j in zip(scores['compound'], scores['score'])}

# Define pockets, compounds and proteins
POCKETS = sorted(DOCKING_RESULTS_ORIGINAL)
COMPOUNDS = sorted(DOCKING_RESULTS_ORIGINAL[POCKETS[0]])
PROTEINS = sorted(set([i.split("_")[1] for i in POCKETS]))
print(f"Number of pockets: {len(POCKETS)}")
print(f"Number of compounds: {len(COMPOUNDS)}")
print(f"Number of proteins: {len(PROTEINS)}")

100%|██████████| 276/276 [00:06<00:00, 40.37it/s] 

Number of pockets: 276
Number of compounds: 100154
Number of proteins: 21


In [ ]:
# Get minimum docking scores per protein
DOCKING_RESULTS_ORIGINAL_PROTEINS = {i: defaultdict(int) for i in PROTEINS}
for pocket in tqdm(POCKETS):
    protein = pocket.split("_")[1]
    for cpd in sorted(DOCKING_RESULTS_ORIGINAL[pocket]):
        DOCKING_RESULTS_ORIGINAL_PROTEINS[protein][cpd] = min(DOCKING_RESULTS_ORIGINAL_PROTEINS[protein][cpd], DOCKING_RESULTS_ORIGINAL[pocket][cpd])

100%|██████████| 276/276 [00:03<00:00, 70.03it/s] 


In [ ]:
SCORE_CUTOFF = -5

proteins = sorted(set([p.split("_")[1] for p in DOCKING_RESULTS_ORIGINAL]))
proteins_to_hits = {p: set() for p in proteins}

for pocket in sorted(DOCKING_RESULTS_ORIGINAL):
    p = pocket.split("_")[1]
    hits = [cpd for cpd, score in DOCKING_RESULTS_ORIGINAL[pocket].items() if score < SCORE_CUTOFF]
    proteins_to_hits[p].update(hits)

def get_hit_overlap(h1, h2):
    return len(h1.intersection(h2))

HIT_OVERLAP = {}
for c1, p1 in enumerate(proteins):
    for p2 in proteins[c1:]:
        ov = get_hit_overlap(proteins_to_hits[p1], proteins_to_hits[p2])
        HIT_OVERLAP[(p1, p2)] = ov
        HIT_OVERLAP[(p2, p1)] = ov

row_names = proteins
matrix = [[HIT_OVERLAP[(i, j)] for j in row_names] for i in row_names]
matrix_df = pd.DataFrame(matrix, index=row_names, columns=row_names)
np.fill_diagonal(matrix_df.values, 0)
node_strength = matrix_df.sum(axis=0) + matrix_df.sum(axis=1)
order = node_strength.sort_values(ascending=False).index
matrix_df = matrix_df.loc[order, order]
matrix_df.values[np.tril_indices_from(matrix_df, k=1)] = 0
matrix_df = matrix_df.rename(index=uniprot_to_gene).rename(columns=uniprot_to_gene)

cmap_dict = {
    'argS':  '#50285A',
    'alaS':  '#FAD782',
    'aspS':  '#FAA08B',
    'gatA':  '#DC9FDC',
    'gatB':  '#AA96FA',
    'thrS':  '#8DC7FA',
    'ileS':  '#BEE6B4',
    'valS':  '#D2D2D2',
    'leuS':  '#F26B38',
    'gltS':  '#4D9DE0',
    'proS':  '#6CC551',
    'glyS':  '#FFC857',
    'serS':  '#E76F51',
    'cysS1': '#2A9D8F',
    'metS':  '#F4A259',
    'lysS':  '#8A5082',
    'hisS':  '#247BA0',
    'trpS':  '#70C1B3',
    'pheS':  '#F25F5C',
    'pheT':  '#FFE066',
    'tyrS':  '#1B9C85'
}

circos = Circos.chord_diagram(
    matrix_df,
    space=0,
    cmap=cmap_dict,
    label_kws=dict(size=12),
    link_kws=dict(ec="k", lw=0.5, alpha=0.8))

circos.plotfig()